# Arquitecturas de Referencia y Patrones Avanzados

## Objetivos

- Dominar **arquitecturas de referencia** empresariales
- Comparar **tradicionales vs modernas** con trade-offs
- Implementar **Data Mesh** y descentralización
- Diseñar **data as a product** a escala
- Evaluar decisiones arquitectónicas con **ADRs**

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Observabilidad, Lineage y Automatización (Senior)](03_observabilidad_lineage_automatizacion.ipynb) | Siguiente: [Alineación Estratégica con Modelos de Dominio, Ontologías y Taxonomías](05_alineacion_estrategica_modelos_dominio.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Evolución de Arquitecturas de Datos
3. 2. Comparativa Detallada: Tradicional vs Moderna
4. 3. Data Mesh: Arquitectura Descentralizada
5. 4. Architecture Decision Records (ADRs)
6. Status
7. Context
8. Decision

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Arquitecturas de Referencia y Patrones Avanzados"] --> C["Notebook siguiente"]
```


## 1. Evolución de Arquitecturas de Datos

### Timeline Histórico

```
1990s              2000s              2010s              2020s
  │                  │                  │                  │
Data Warehouse ────▶ Data Lake ───────▶ Lakehouse ───────▶ Data Mesh
  │                  │                  │                  │
• ETL batch         • Schema-on-read   • ACID on lake    • Domain-oriented
• Esquema rígido    • Big Data         • Delta/Iceberg   • Self-serve
• Kimball/Inmon     • Hadoop/Spark     • Unified          • Data as product
```

### Arquitecturas Tradicionales

#### A. Data Warehouse Tradicional (Inmon)

**Características:**
- Centralizado, normalizado (3NF)
- ETL batch diario/semanal
- Data marts dimensional por departamento
- Esquema rígido, definido antes de carga

**✅ Ventajas:**
- Alta calidad y consistencia
- Ideal para reporting estructurado
- ACID transactions

**❌ Limitaciones:**
- Inflexible (schema changes costosos)
- No escala para Big Data
- Latencia alta (batch)

#### B. Data Lake (Hadoop Era)

**Características:**
- Schema-on-read (flexibilidad)
- Almacena datos crudos (cualquier formato)
- Procesamiento distribuido (Spark, Hive)
- Escalabilidad horizontal

**✅ Ventajas:**
- Escala a petabytes
- Soporta datos no estructurados
- Bajo costo de almacenamiento

**❌ Limitaciones:**
- "Data Swamp" (desorden)
- Sin ACID, difícil garantizar calidad
- Performance inconsistente

### Arquitecturas Modernas

#### C. Lakehouse

**Concepto:** Combina lo mejor de Warehouse (ACID, performance) y Lake (flexibilidad, escala).

**Tecnologías:**
- Delta Lake (Databricks)
- Apache Iceberg (Netflix)
- Apache Hudi (Uber)

**Capacidades:**
- ACID transactions en object storage (S3, ADLS)
- Time travel, schema evolution
- Merge/Update/Delete eficientes
- Metadatos centralizados

#### D. Data Mesh

**Concepto:** Descentralización orientada a dominios, data as a product.

**Principios:**
1. **Domain ownership:** Cada dominio gestiona sus datos
2. **Data as a product:** Calidad, SLAs, APIs
3. **Self-serve platform:** Infraestructura común
4. **Federated governance:** Políticas compartidas, ejecución local

```
                 Federated Governance
                 (Policies, Standards)
                          │
       ┌──────────────────┼──────────────────┐
       │                  │                  │
   Marketing         Finance            Sales
   Domain            Domain             Domain
       │                  │                  │
   [Data Product]    [Data Product]    [Data Product]
   • Customer 360    • Revenue KPIs    • Pipeline Metrics
   • Campaigns       • Forecasts       • Opportunities
       │                  │                  │
       └──────────────────┴──────────────────┘
                          │
              Self-Serve Data Platform
              (Compute, Storage, Catalog)
```

## 2. Comparativa Detallada: Tradicional vs Moderna

### Tabla Comparativa

| Dimensión | Data Warehouse | Data Lake | Lakehouse | Data Mesh |
|-----------|----------------|-----------|-----------|------------|
| **Esquema** | Schema-on-write | Schema-on-read | Híbrido | Por dominio |
| **Governance** | Centralizado | Débil | Centralizado | Federado |
| **Ownership** | Equipo central | Equipo central | Equipo central | Dominios |
| **Escalabilidad** | Vertical | Horizontal | Horizontal | Horizontal |
| **Performance** | Alto (SQL) | Variable | Alto | Variable |
| **Calidad** | Alta | Baja-Media | Alta | Alta (por contrato) |
| **Flexibilidad** | Baja | Alta | Alta | Alta |
| **Costo** | Alto | Bajo | Medio | Medio-Alto |
| **Complejidad** | Media | Alta | Alta | Muy Alta |
| **Cuándo usar** | BI estructurado | Big Data, ML | Unificado | Org grandes |

### Trade-offs Clave

#### Centralizado vs Descentralizado

**Centralizado (DW, Lake, Lakehouse):**
- ✅ Consistencia global
- ✅ Governance simplificado
- ❌ Bottleneck en equipo central
- ❌ No escala para múltiples dominios

**Descentralizado (Data Mesh):**
- ✅ Autonomía de dominios
- ✅ Escala organizacionalmente
- ❌ Riesgo de inconsistencia
- ❌ Requiere madurez cultural

#### Batch vs Streaming vs Híbrido

| Patrón | Latencia | Complejidad | Costo | Caso de Uso |
|--------|----------|-------------|-------|-------------|
| **Batch puro** | Horas/días | Baja | Bajo | Reportes históricos |
| **Streaming puro** | Segundos | Alta | Alto | Alertas real-time |
| **Lambda** | Ambas | Muy Alta | Alto | Requiere ambas vistas |
| **Kappa** | Baja | Media | Medio | Todo desde streaming |

In [ ]:
# Framework de decisión arquitectónica
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

class ArchitectureDecisionFramework:
    """Framework para evaluar arquitecturas según contexto"""
    
    def __init__(self):
        # Definir criterios de evaluación (peso 0-1)
        self.criterios = {
            'scalability': 0.9,
            'performance': 0.8,
            'flexibility': 0.7,
            'governance': 0.8,
            'cost': 0.6,
            'complexity': 0.5,
            'time_to_market': 0.7
        }
        
        # Evaluar arquitecturas (score 0-10)
        self.arquitecturas = {
            'Data Warehouse': {
                'scalability': 4,
                'performance': 9,
                'flexibility': 3,
                'governance': 9,
                'cost': 4,
                'complexity': 6,
                'time_to_market': 5
            },
            'Data Lake': {
                'scalability': 9,
                'performance': 6,
                'flexibility': 9,
                'governance': 4,
                'cost': 8,
                'complexity': 7,
                'time_to_market': 6
            },
            'Lakehouse': {
                'scalability': 9,
                'performance': 8,
                'flexibility': 8,
                'governance': 7,
                'cost': 6,
                'complexity': 8,
                'time_to_market': 4
            },
            'Data Mesh': {
                'scalability': 10,
                'performance': 7,
                'flexibility': 10,
                'governance': 8,
                'cost': 5,
                'complexity': 10,
                'time_to_market': 3
            }
        }
    
    def calcular_score_ponderado(self):
        """Calcula score total ponderado por criterio"""
        resultados = {}
        
        for arq, scores in self.arquitecturas.items():
            score_total = sum(
                scores[criterio] * peso 
                for criterio, peso in self.criterios.items()
            ) / sum(self.criterios.values())
            resultados[arq] = round(score_total, 2)
        
        return resultados
    
    def recomendar(self, contexto):
        """Recomienda arquitectura según contexto"""
        recomendaciones = {
            'startup_mvp': 'Data Warehouse',
            'big_data_analytics': 'Data Lake',
            'unified_platform': 'Lakehouse',
            'large_org_multiple_domains': 'Data Mesh'
        }
        return recomendaciones.get(contexto, 'Lakehouse')
    
    def visualizar_comparacion(self):
        """Genera radar chart comparativo"""
        criterios_list = list(self.criterios.keys())
        num_criterios = len(criterios_list)
        
        # Preparar datos
        angles = np.linspace(0, 2 * np.pi, num_criterios, endpoint=False).tolist()
        angles += angles[:1]  # Cerrar el círculo
        
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
        
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
        
        for idx, (arq, scores) in enumerate(self.arquitecturas.items()):
            values = [scores[c] for c in criterios_list]
            values += values[:1]
            ax.plot(angles, values, 'o-', linewidth=2, label=arq, color=colors[idx])
            ax.fill(angles, values, alpha=0.15, color=colors[idx])
        
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(criterios_list)
        ax.set_ylim(0, 10)
        ax.set_title('Comparación de Arquitecturas de Datos', size=16, weight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        ax.grid(True)
        
        plt.tight_layout()
        plt.show()

# Ejecutar análisis
framework = ArchitectureDecisionFramework()

print("=" * 70)
print("FRAMEWORK DE DECISIÓN ARQUITECTÓNICA")
print("=" * 70)

# Scores ponderados
scores = framework.calcular_score_ponderado()
print("\n📊 Scores Ponderados:")
for arq, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {arq:20s}: {score:.2f}/10")

# Recomendaciones por contexto
print("\n💡 Recomendaciones por Contexto:")
contextos = {
    'startup_mvp': 'Startup con MVP (2-5 personas)',
    'big_data_analytics': 'Big Data Analytics (ML, AI)',
    'unified_platform': 'Plataforma unificada empresarial',
    'large_org_multiple_domains': 'Organización grande (>1000 empleados, múltiples dominios)'
}

for ctx, desc in contextos.items():
    rec = framework.recomendar(ctx)
    print(f"  {desc:50s} → {rec}")

# Visualizar
framework.visualizar_comparacion()

## 3. Data Mesh: Arquitectura Descentralizada

### Anatomía de un Data Product

Un **data product** en Data Mesh tiene:

```yaml
data_product:
  metadata:
    name: customer-360
    domain: marketing
    owner_team: customer-analytics
    version: 3.2.0
  
  interfaces:
    - type: sql
      endpoint: gold.marketing.customer_360
    - type: rest_api
      endpoint: https://api.company.com/data/customer-360
    - type: streaming
      topic: customer.events
  
  sla:
    freshness: 15 minutes
    availability: 99.9%
    response_time_p95: 500ms
    quality:
      completeness: 98%
      accuracy: 99%
  
  schema:
    version: 3.2.0
    compatibility: backward
    changelog:
      - v3.2.0: Added lifetime_value field
      - v3.1.0: Renamed segment to customer_segment
  
  lineage:
    sources:
      - crm.contacts
      - sales.orders
      - support.tickets
  
  consumers:
    - sales-team (forecast models)
    - support-team (customer service)
    - finance-team (revenue analysis)
```

### Implementación de Self-Serve Platform

**Componentes:**

1. **Data Product Catalog:** Discovery, documentation
2. **Compute Mesh:** Spark/Flink on Kubernetes, auto-scaling
3. **Storage Mesh:** Object storage (S3) + metadata (Hive/Iceberg)
4. **Access Control:** RBAC/ABAC centralizado
5. **Observability:** Logs, metrics, lineage
6. **CI/CD Pipelines:** Deploy automatizado de data products

### Federated Governance

**Global Policies (enforced centrally):**
- Clasificación de datos (PII, confidencial)
- Retención mínima/máxima
- Estándares de seguridad (encriptación)
- SLAs mínimos de calidad

**Local Implementation (domain-specific):**
- Cómo lograr calidad (validations específicas)
- Qué herramientas usar (Spark vs Flink)
- Frecuencia de actualización
- Esquemas y transformaciones

In [ ]:
# Simulador de Data Mesh Registry
import json
from datetime import datetime, timedelta

class DataProductRegistry:
    """Registro centralizado de data products"""
    
    def __init__(self):
        self.products = {}
        self.metrics = {}
    
    def register_product(self, product_spec):
        """Registrar un data product"""
        product_id = f"{product_spec['domain']}.{product_spec['name']}"
        self.products[product_id] = {
            **product_spec,
            'registered_at': datetime.now().isoformat(),
            'status': 'active'
        }
        print(f"✅ Data Product registrado: {product_id} v{product_spec['version']}")
        return product_id
    
    def discover(self, domain=None, tag=None):
        """Descubrir data products"""
        results = []
        for pid, product in self.products.items():
            if domain and product['domain'] != domain:
                continue
            if tag and tag not in product.get('tags', []):
                continue
            results.append({
                'id': pid,
                'name': product['name'],
                'domain': product['domain'],
                'owner': product['owner_team'],
                'version': product['version']
            })
        return results
    
    def track_usage(self, product_id, consumer, query_count, avg_latency_ms):
        """Trackear métricas de uso"""
        if product_id not in self.metrics:
            self.metrics[product_id] = []
        
        self.metrics[product_id].append({
            'timestamp': datetime.now().isoformat(),
            'consumer': consumer,
            'query_count': query_count,
            'avg_latency_ms': avg_latency_ms
        })
    
    def generate_health_report(self, product_id):
        """Generar reporte de salud del producto"""
        if product_id not in self.products:
            return None
        
        product = self.products[product_id]
        metrics = self.metrics.get(product_id, [])
        
        # Calcular estadísticas
        total_queries = sum(m['query_count'] for m in metrics)
        unique_consumers = len(set(m['consumer'] for m in metrics))
        avg_latency = sum(m['avg_latency_ms'] for m in metrics) / len(metrics) if metrics else 0
        
        # Evaluar cumplimiento SLA
        sla_latency = product['sla']['response_time_p95_ms']
        sla_met = avg_latency < sla_latency
        
        return {
            'product': product_id,
            'health': 'healthy' if sla_met else 'degraded',
            'metrics': {
                'total_queries': total_queries,
                'unique_consumers': unique_consumers,
                'avg_latency_ms': round(avg_latency, 2)
            },
            'sla_compliance': {
                'latency_target': sla_latency,
                'latency_actual': round(avg_latency, 2),
                'status': '✅ CUMPLE' if sla_met else '❌ INCUMPLE'
            }
        }

# Crear registry
registry = DataProductRegistry()

print("=" * 70)
print("SIMULACIÓN DE DATA MESH")
print("=" * 70 + "\n")

# Registrar data products
customer_360 = registry.register_product({
    'name': 'customer_360',
    'domain': 'marketing',
    'owner_team': 'customer-analytics',
    'version': '3.2.0',
    'tags': ['customer', 'analytics', 'pii'],
    'sla': {
        'freshness_minutes': 15,
        'availability_pct': 99.9,
        'response_time_p95_ms': 500
    }
})

revenue_kpis = registry.register_product({
    'name': 'revenue_kpis',
    'domain': 'finance',
    'owner_team': 'financial-planning',
    'version': '2.1.0',
    'tags': ['finance', 'kpis', 'executive'],
    'sla': {
        'freshness_minutes': 60,
        'availability_pct': 99.95,
        'response_time_p95_ms': 1000
    }
})

# Simular uso
print("\n📊 Simulando uso de data products...")
registry.track_usage(customer_360, 'sales-team', query_count=1500, avg_latency_ms=350)
registry.track_usage(customer_360, 'support-team', query_count=800, avg_latency_ms=420)
registry.track_usage(revenue_kpis, 'executive-dashboard', query_count=200, avg_latency_ms=650)

# Descubrir products
print("\n🔍 Descubriendo data products en dominio 'marketing':")
products = registry.discover(domain='marketing')
for p in products:
    print(f"  • {p['id']} v{p['version']} (owner: {p['owner']})")

# Health reports
print("\n🏥 Reportes de Salud:\n")
for pid in [customer_360, revenue_kpis]:
    report = registry.generate_health_report(pid)
    print(f"{'='*70}")
    print(f"Data Product: {report['product']}")
    print(f"Estado: {report['health'].upper()}")
    print(f"\nMétricas:")
    print(f"  Queries totales: {report['metrics']['total_queries']:,}")
    print(f"  Consumidores únicos: {report['metrics']['unique_consumers']}")
    print(f"  Latencia promedio: {report['metrics']['avg_latency_ms']} ms")
    print(f"\nSLA Compliance:")
    print(f"  Target: < {report['sla_compliance']['latency_target']} ms")
    print(f"  Actual: {report['sla_compliance']['latency_actual']} ms")
    print(f"  Status: {report['sla_compliance']['status']}")
    print()

## 4. Architecture Decision Records (ADRs)

### ¿Qué son los ADRs?

**Architecture Decision Records** documentan decisiones arquitectónicas importantes con:
- **Contexto:** ¿Por qué se necesita una decisión?
- **Decisión:** ¿Qué se decidió?
- **Consecuencias:** Trade-offs, pros/cons
- **Alternativas consideradas:** ¿Qué más se evaluó?

### Ejemplo: ADR-001

```markdown
# ADR-001: Adoptar Lakehouse Architecture con Delta Lake

## Status
Accepted | 2025-11-07

## Context
Necesitamos unificar nuestro Data Warehouse (Redshift) y Data Lake (S3) 
que actualmente están desconectados, causando:
- Duplicación de datos y ETL
- Inconsistencias entre sistemas
- Altos costos operacionales (2 equipos, 2 stacks)
- Latencia en sincronización (6 horas)

## Decision
Migraremos a Lakehouse architecture usando Delta Lake on S3:
- Storage: S3 (mantener)
- Formato: Delta Lake (ACID on Parquet)
- Compute: Databricks + Spark
- Catalog: Unity Catalog

## Consequences

### Positive
- Elimina duplicación: un solo storage
- ACID transactions en S3
- Time travel para auditoría
- Menor costo: decommissionar Redshift
- Unifica batch y streaming

### Negative
- Migración compleja (6-9 meses)
- Requiere re-training de equipo
- Vendor lock-in con Databricks
- Performance inicial puede ser menor que Redshift

### Risks
- Downtime durante migración (mitigation: blue/green)
- Queries legacy pueden requerir reescritura
- Costo de Databricks puede exceder estimado

## Alternatives Considered

### 1. Mantener status quo (Redshift + S3)
- ❌ Rechazado: Duplicación continúa
- ❌ Costos crecen linealmente

### 2. Migrar todo a Redshift
- ❌ Rechazado: No escala para Big Data
- ❌ Costo prohibitivo para PBs

### 3. Snowflake Data Cloud
- ⚖️  Alternativa viable
- ❌ Rechazado: Mayor costo que Delta
- ❌ Menos control sobre storage

### 4. Apache Iceberg on S3
- ⚖️  Alternativa viable (open source)
- ❌ Rechazado: Menos maduro, comunidad más pequeña

## Implementation Plan
Ver [Migration Roadmap](migration-roadmap.md)

## References
- Delta Lake RFC
- Cost analysis spreadsheet
- POC results (Q3 2025)
```

## 5. Casos de Estudio de Migraciones Reales

### 5.1 Netflix: Hive Metastore + Parquet → Apache Iceberg
**Contexto inicial:** Gran catálogo Hive sobre miles de tablas Parquet, problemas de schema evolution y borrados.
**Problemas:**
- No soporte nativo para `DELETE`/`UPDATE` eficientes.
- Dificultad para time travel y auditoría precisa.
- Metadatos fragmentados, operaciones de mantenimiento costosas.
**Objetivos:** Simplificar gestión de metadatos, habilitar operaciones ACID, mejorar confiabilidad de jobs.
**Estrategia de migración (fases):**
1. Inventario y clasificación de tablas (críticas vs analíticas).
2. POC de tablas candidatas en Iceberg con dual-write (Parquet + Iceberg).
3. Validación de consistencia (row counts, checksums, schema).
4. Activación de lectura Iceberg para consumidores no críticos.
5. Cut-over total y deshabilitación de parquet legacy.
**Resultados/Métricas:**
- Reducción 35% tiempos de planificación de queries (metadatos más compactos).
- Time travel habilitado para auditorías regulatorias (<5 min recuperación).
- Menos fallos de jobs de mantenimiento (particiones obsoletas) → -40% tickets.
**Lecciones clave:** Empezar por tablas de baja criticidad, automatizar verificación de equivalencia.

### 5.2 Uber: Hadoop/Hive → Apache Hudi (Incremental Ingestion)
**Contexto inicial:** Procesos batch pesados nocturnos sobre grandes datasets transaccionales.
**Problemas:**
- Latencia alta (datos frescos solo cada 24h).
- Reprocesos costosos ante fallas.
- Dificultad para upserts granulares.
**Objetivos:** Ingesta incremental continua, upserts y borrados eficientes, menor latencia.
**Estrategia:**
1. Identificar tablas con alto churn (actualizaciones frecuentes).
2. Implementar pipeline CDC → Kafka → Ingesta Hudi (copy-on-write).
3. Mantener vistas Hive compatibles (query engine transición suave).
4. Optimizar clustering y compaction schedule.
5. Expandir a tablas de baja frecuencia.
**Resultados:** Latencia de disponibilidad de datos ↓ de 24h a ~30min. Reprocesos masivos casi eliminados. Costos de almacenamiento estables por mejor manejo de versiones.
**Lecciones:** Ajustar compaction temprano para evitar degradación de lectura.

### 5.3 Airbnb: S3 Raw + Hive ETL → Delta Lake Unificado (Lakehouse)
**Contexto inicial:** Data Lake desordenado; duplicación en capas intermedias, pipelines frágiles.
**Problemas:**
- Inconsistencias entre capas derivadas.
- Dificultad en auditoría (sin time travel nativo).
- Costo operativo alto (scripts personalizados de housekeeping).
**Objetivos:** Unificar pipeline, ACID, gobernanza central (catalog + permisos), facilitar ML.
**Fases:**
1. Definir modelo de zonas Bronze/Silver/Gold con contratos de calidad.
2. Migrar tablas de dimensión críticas a Delta (dual-write y validación).
3. Implementar data quality + expectations automatizadas antes de promoción.
4. Integrar ML feature pipelines consumiendo Silver.
5. Decomisionar rutas Hive no estandarizadas.
**Métricas de éxito:**
- Tiempo de onboarding de nuevos analistas ↓ 50%.
- Fallos de jobs ETL ↓ 45%.
- Reuso de datasets ML ↑ 30%.
**Lecciones:** Formalizar nomenclatura y versionado antes de migrar para evitar caos semántico.

### 5.4 Spotify: Pipelines ETL Redshift → Lakehouse en GCP (BigQuery + Storage + Dataflow)
**Contexto inicial:** Crecimiento de volumen y necesidad de procesar eventos near real-time para recomendaciones.
**Problemas:**
- Limitaciones de escalado vertical en cluster inicial.
- Costos altos por almacenamiento caliente redundante.
- Latencia en incorporación de nuevos tipos de eventos.
**Objetivos:** Elasticidad, separación compute/storage, integración streaming nativa.
**Fases:**
1. Replicación continua (DMS / Dataflow) de datos relacionales a BigQuery.
2. Reescritura de transformaciones batch a Dataflow + SQL modular.
3. Implementación de particionado y clustering para performance.
4. Enriquecimiento streaming (Pub/Sub) para features en tiempo real.
5. Optimización de costo (tabla tiering + control de retención).
**Resultados:**
- Aceleración de experimentación de modelos (ciclos entrenamiento ↓ 25%).
- Optimización de costo almacenamiento frío (archivado a GCS Nearline).
- Latencia de incorporación de nuevo evento: semanas → días.
**Lecciones:** Definir convención estricta de particionado; involucrar FinOps desde diseño inicial.

## 6. Métricas de Éxito y KPIs de Migración
| Categoría | KPI | Definición | Objetivo de referencia |
|-----------|-----|------------|------------------------|
| Técnica | Latencia de disponibilidad | Tiempo desde ingesta a disponibilidad en capa objetivo | < 1h (operacional), < 24h histórico |
| Técnica | Fallos de pipeline | % ejecuciones fallidas post-migración | < 2% |
| Calidad | Drift semántico | Cambios no documentados en esquema vs total cambios | < 5% |
| Operativa | Tickets de soporte | Incidencias mensuales relacionadas a datos | -30% vs baseline |
| Governance | Cumplimiento de contratos | % datasets con contratos de datos vigentes | > 90% |
| Productividad | Onboarding analista | Tiempo para producir primer dashboard | -40% vs baseline |
| Reuso | Datasets reutilizados | % datasets usados por >1 dominio | > 50% |
| FinOps | Costo por TB procesado | Compute + storage / TB mensual | -15% vs baseline |

## 7. Lecciones Aprendidas en Migraciones
**Patrones exitosos:**
- Dual-write + verificación automática (checksums, row counts) para confianza.
- Catalogación temprana (metadatos + clasificación) evita data swamp 2.0.
- Contratos de datos (schema + SLAs) antes de permitir consumo externo.
- Feature toggles para activar progresivamente nuevas rutas.
- Métricas de humo (smoke) en primera semana tras cut-over.
**Errores comunes:**
- Migrar todo al inicio (big bang) → alto riesgo.
- Subestimar costo de reentrenar equipo (skills gap).
- No alinear SLAs antiguos vs nuevos (percepción de degradación).
- Ignorar compatibilidad descendente en APIs SQL.
- Falta de rollback plan documentado.

## 8. Plantilla Completa ADR (Generalizada)
```markdown
# ADR-XXX: <Título Conciso>

## Status
Proposed | Accepted | Deprecated | Superseded <fecha>

## Context
Antecedentes, drivers (ej. crecimiento volumen, latencia, compliance). Datos cuantitativos de baseline.

## Forces / Drivers
- Escalabilidad
- Costo
- Latencia
- Calidad
- Seguridad / Cumplimiento
- Time-to-market

## Decision
Descripción precisa (incluye componentes seleccionados, alcance, fuera de alcance).

## Options Considered
| Opción | Ventajas | Desventajas | Costo relativo | Riesgos |
|--------|----------|-------------|----------------|---------|
| A | ... | ... | Medio | ... |
| B | ... | ... | Alto | ... |
| C | ... | ... | Bajo | ... |

## Rationale
¿Por qué la opción elegida es superior dadas las fuerzas?

## Consequences
### Positivas
- ...
### Negativas
- ...

## Risks & Mitigations
| Riesgo | Impacto | Probabilidad | Mitigación |
|--------|---------|-------------|------------|
| X | Alto | Media | ... |

## Compliance & Privacy Impact
Regulaciones afectadas, nuevos controles requeridos.

## Cost Impact
Estimación OPEX/CAPEX, horizonte de amortización.

## Operational Impact
Cambios en monitoreo, alertas, procesos oncall.

## Sunset / Retirement Plan
Cómo se desactiva la solución futura o se migra a otra.

## Review Date
Fecha para reevaluar (ej. 12 meses). Criterios de éxito.

## References
Links a POCs, benchmarks, estimaciones costo, discusiones previas.
```

## 9. Checklist de Evaluación Arquitectónica Integral
**Categorías:** Escalabilidad, Rendimiento, Disponibilidad, Resiliencia, Seguridad, Gobernanza, Calidad de Datos, Observabilidad, Costos (FinOps), Experiencia de Desarrollo, Sostenibilidad, Cumplimiento.

### 9.1 Ítems
| Categoría | Ítem | Estado (✓/✗/NA) |
|-----------|------|----------------|
| Escalabilidad | Soporta crecimiento 3× volumen en 12 meses | ✓ |
| Rendimiento | Latencia p95 acorde SLAs | ✓ |
| Disponibilidad | DR/Backup probado | ✗ |
| Resiliencia | Circuit breakers definidos | ✓ |
| Seguridad | Clasificación PII aplicada | ✓ |
| Gobernanza | Linaje completo crítico | ✓ |
| Calidad | Contratos de datos ≥90% datasets | ✗ |
| Observabilidad | Métricas negocio + técnicas en dashboards | ✓ |
| FinOps | Tagging costo por dominio | ✓ |
| DevExp | Tiempo deploy <30 min | ✓ |
| Sostenibilidad | Costo idle <10% gasto compute | ✗ |
| Cumplimiento | GDPR/retención documentado | ✓ |

### 9.2 Puntaje
Asignar pesos por categoría y calcular score compuesto.
```python
architecture_score = {
  'scalability': 0.85,
  'performance': 0.80,
  'resilience': 0.75,
  'security': 0.90,
  'governance': 0.70,
  'data_quality': 0.65,
  'observability': 0.80,
  'finops': 0.60,
  'dev_experience': 0.78,
  'compliance': 0.88
}
final_score = round(sum(architecture_score.values()) / len(architecture_score), 2)
print('Architecture Composite Score:', final_score)
```
**Interpretación:**
- >0.80 Excelente
- 0.70–0.79 Adecuado con mejoras puntuales
- <0.70 Requiere plan de fortalecimiento

### 9.3 Acciones Correctivas
- Priorizar contratos de datos (calidad) próximos 90 días.
- Implementar DR test automatizado trimestral.
- Reducir idle compute mediante autoscaling y scheduling.


## 10. Ejercicio: Diseño de Arquitectura Empresarial

**Escenario:** Banco multinacional con 50M clientes

**Requisitos:**
- **Dominios:** Retail Banking, Corporate Banking, Investment, Risk, Compliance
- **Volumen:** 10 PB datos históricos, 500 TB/mes crecimiento
- **Velocidad:** Transacciones real-time, reportes regulatorios batch
- **Regulación:** GDPR, BCBS 239 (risk data aggregation)
- **Equipos:** 150+ data engineers/analysts distribuidos globalmente

**Tareas:**

### 1. Selección de Arquitectura (25%)
- Evaluar: Data Warehouse, Data Lake, Lakehouse, Data Mesh
- Justificar elección con 5 criterios ponderados
- Identificar riesgos y mitigaciones

### 2. Diseño de Alto Nivel (30%)
- Diagramar arquitectura completa
- Definir capas (Bronze/Silver/Gold por dominio)
- Especificar tecnologías (storage, compute, catalog, governance)
- Diseñar 3 data products clave

### 3. Governance Federado (20%)
- Definir políticas globales (5 mínimo)
- Especificar ownership por dominio
- Diseñar proceso de aprobación de data products
- Crear matriz RACI para governance

### 4. ADR Crítico (15%)
- Documentar decisión arquitectónica más importante
- Incluir: contexto, decisión, consecuencias, alternativas
- Identificar riesgos y plan de mitigación

### 5. Plan de Implementación (10%)
- Roadmap por fases (18 meses)
- Quick wins (3 meses)
- Métricas de éxito
- Presupuesto estimado


## 11. Resumen Ejecutivo

### Evolución Arquitectónica

**1990s-2000s:** Data Warehouse
- Centralizado, schema-on-write
- ✅ Calidad, ❌ Flexibilidad

**2010s:** Data Lake
- Escalable, schema-on-read
- ✅ Flexibilidad, ❌ Calidad

**2020s:** Lakehouse
- Unificado, ACID en lake
- ✅ Equilibrio, ❌ Complejidad

**Futuro:** Data Mesh
- Descentralizado, domain-oriented
- ✅ Escala organizacional, ❌ Coordinación

### Decisiones Clave

**Centralizado vs Descentralizado:**
- < 50 personas → Centralizado
- > 100 personas, múltiples dominios → Data Mesh

**Batch vs Streaming:**
- Reportes históricos → Batch
- Alertas real-time → Streaming
- Ambos → Lambda (2 pipelines) o Kappa (1 pipeline)

**Build vs Buy:**
- Open source (Delta, Iceberg) → Más control, más trabajo
- Managed (Snowflake, Databricks) → Menos trabajo, mayor costo

### Herramientas Clave

**Data Products:** Documentar con data contracts, SLAs
**ADRs:** Registrar decisiones importantes
**Federated Governance:** Balance autonomía y consistencia
**Self-Serve Platform:** Reducir time-to-market
**Checklists & KPIs:** Guiar evaluación y medir progreso

### Próximos Pasos Recomendados
1. Completar inventario y clasificación de datasets críticos.
2. Definir contratos de datos para top 20 datasets.
3. Ejecutar POC de migración incremental (dual-write) en zona controlada.
4. Implementar dashboard unificado de KPIs de migración.
5. Formalizar comité de federated governance y cadencia mensual.

### 📚 Referencias
- "Data Mesh" - Zhamak Dehghani (O'Reilly)
- "Fundamentals of Data Engineering" - Joe Reis & Matt Housley
- Delta Lake / Apache Iceberg / Apache Hudi docs
- ThoughtWorks Technology Radar
- Martin Fowler's blog


## 12. Ejercicio Transversal: Migración & Evaluación KPI (Nivel Senior)

Este ejercicio integra decisión arquitectónica, KPIs de migración y gobernanza aplicada usando el dataset `customers_dim.csv` (ruta: `../04-recursos/datasets/customers_dim.csv`).

### Contexto
Una organización con un Data Warehouse tradicional (on-prem) planea migrar a un Lakehouse gestionado. El objetivo es mejorar time-to-data para analítica avanzada y reducir costos de mantenimiento sin sacrificar gobernanza.

### Dataset: customers_dim.csv
Campos: customer_id, name, segment, country, signup_date, is_active.
Anomalías intencionales: filas duplicadas, segment faltante, fechas fuera de rango, inconsistencias en is_active vs signup_date.

### Objetivo General
1. Evaluar la viabilidad de migración (Lakehouse vs Hybrid) con criterios cuantitativos.
2. Construir un KPI baseline y metas 12 meses.
3. Redactar ADR de migración con riesgos y mitigaciones.
4. Definir plan incremental (dual-write + shadow reads) y gobernanza federada mínima.

### KPIs Propuestos
| KPI | Definición | Baseline | Meta 12m | Observaciones |
|-----|------------|----------|----------|---------------|
| Time-to-Data (hrs) | Tiempo desde ingestión a disponibilidad analítica | 24 | 2 | Streaming + incremental processing |
| Data Quality Score (%) | % reglas críticas cumplidas | 88 | 97 | Contratos + validación automatizada |
| Cost Efficiency ($/TB) | Costo operativo por TB procesado | 180 | 110 | Optimización storage + spot compute |
| Governance Coverage (%) | % datasets con contrato + lineage | 40 | 85 | Plataforma self-serve + catálogo |
| Deployment Frequency (/mes) | Releases de pipeline productivos | 2 | 12 | CI/CD maduro + test automatizados |

### Tareas
1. Cargar dataset y cuantificar anomalías (tabla tipo/conteo/%). 
2. Estimar impacto de cada anomalía sobre Data Quality Score.
3. Proponer 5 decisiones clave (storage, formato, orquestación, catálogo, seguridad) con alternativa descartada y justificación.
4. Redactar ADR principal (Lakehouse migración) con: Contexto, Decisión, Alternativas, Consecuencias, Riesgos, Mitigación.
5. Definir plan fases (0-3m, 3-6m, 6-12m) indicando hitos y KPIs a monitorear.
6. Diseñar esquema de gobernanza federada: roles, RACI, cadencia, artefactos.
7. Calcular score agregado (ponderaciones) y mostrar mejora esperada (baseline vs target).
8. Lista de riesgos top 5 con probabilidad/impacto/mitigación.

### Ponderaciones Scoring
| Aspecto | Peso |
|---------|------|
| Calidad Datos | 25% |
| Time-to-Data | 20% |
| Gobernanza | 20% |
| Costos | 20% |
| Escalabilidad | 15% |

### Rúbrica Evaluación
| Aspecto | Peso |
|---------|------|
| Cuantificación anomalías y análisis impacto | 20% |
| ADR completo y claro | 25% |
| Plan fases realista + KPIs | 20% |
| Scoring justificado + cálculo | 20% |
| Riesgos y gobernanza federada | 15% |

### Siguientes Pasos (Opcional)
- Simular evolución KPI trimestral.
- Incorporar dashboard Prometheus/Grafana para métricas operativas.
- Añadir control de calidad automatizado con Great Expectations en pipelines incrementales.

### Referencias Internas
- Plantilla transversal: `../04-recursos/plantilla_ejercicio_transversal.md`
- Dataset: `../04-recursos/datasets/customers_dim.csv`

> Este ejercicio cierra el ciclo práctico: Junior (detección), Mid (features & costos), Senior (decisiones estratégicas, migración y gobernanza).


In [ ]:
# Código base para ejercicio transversal Senior (Migración & KPI)
import pandas as pd
from pathlib import Path

DATA_REL = Path("../04-recursos/datasets/customers_dim.csv")
DATA_ROOT = Path("04-recursos/datasets/customers_dim.csv")

if DATA_REL.exists():
    path_customers = DATA_REL
elif DATA_ROOT.exists():
    path_customers = DATA_ROOT
else:
    raise FileNotFoundError("No se encontró customers_dim.csv en rutas esperadas")

df_cust = pd.read_csv(path_customers)
print("Shape:", df_cust.shape)
print(df_cust.head())

# Perfil y anomalías
anomalies = {}
# Duplicados
anomalies['duplicate_rows'] = df_cust.duplicated().sum()
# Segment nulo
anomalies['missing_segment'] = df_cust['segment'].isna().sum() if 'segment' in df_cust.columns else 0
# Fechas fuera de rango (definimos 2015-01-01 a 2025-12-31)
if 'signup_date' in df_cust.columns:
    df_cust['signup_date'] = pd.to_datetime(df_cust['signup_date'], errors='coerce')
    mask_date = (df_cust['signup_date'] < pd.Timestamp('2015-01-01')) | (df_cust['signup_date'] > pd.Timestamp('2025-12-31'))
    anomalies['out_of_range_signup_date'] = mask_date.sum()
else:
    anomalies['out_of_range_signup_date'] = 0
# is_active inconsistente (is_active==True pero signup_date nula)
if 'is_active' in df_cust.columns:
    anomalies['active_missing_date'] = ((df_cust['is_active'] == True) & (df_cust['signup_date'].isna())).sum()
else:
    anomalies['active_missing_date'] = 0

print("\nTabla anomalías:")
print("tipo, conteo, porcentaje")
for k,v in anomalies.items():
    pct = (v / len(df_cust)) * 100 if len(df_cust) else 0
    print(f"{k}, {v}, {pct:.2f}%")

# Impacto estimado sobre Data Quality Score (simple: cada anomalía resta un porcentaje)
base_quality = 88.0  # baseline hipotético
penalties = 0
if anomalies['duplicate_rows'] > 0:
    penalties += 2
if anomalies['missing_segment'] > 0:
    penalties += 3
if anomalies['out_of_range_signup_date'] > 0:
    penalties += 2
if anomalies['active_missing_date'] > 0:
    penalties += 1
quality_estimated = max(base_quality - penalties, 0)
print(f"\nData Quality Score estimado (baseline {base_quality} - penalizaciones {penalties}): {quality_estimated}%")

# Scoring ponderado arquitectura (valores hipotéticos baseline vs target)
weights = {
    'calidad': 0.25,
    'time_to_data': 0.20,
    'gobernanza': 0.20,
    'costos': 0.20,
    'escalabilidad': 0.15
}
baseline_scores = {
    'calidad': quality_estimated/100,   # normalizado
    'time_to_data': 24/24,              # 24h baseline -> factor 1
    'gobernanza': 0.40,                 # 40% datasets con contrato
    'costos': 110/180,                  # eficiencia relativa (menor es mejor -> invertimos: 110 target/180 baseline?) aquí baseline ~0.61
    'escalabilidad': 0.50               # percepción inicial
}
# Ajuste: time_to_data normalizado invertido (menor mejor) => baseline 24h / 24h =1 => convertimos a (1 - (valor-2)/(24-2))
def normalize_time_to_data(hours, min_target=2, max_baseline=24):
    ratio = (hours - min_target) / (max_baseline - min_target)
    return max(0, 1 - ratio)
baseline_scores['time_to_data'] = normalize_time_to_data(24)

# Target esperado
target_scores = {
    'calidad': 0.97,
    'time_to_data': normalize_time_to_data(2),
    'gobernanza': 0.85,
    'costos': 1.0,  # asumimos mejora eficiencia (proxy)
    'escalabilidad': 0.80
}

arch_baseline = sum(baseline_scores[k]*weights[k] for k in weights)
arch_target = sum(target_scores[k]*weights[k] for k in weights)
print(f"\nArchitecture Weighted Score baseline: {arch_baseline:.2f}")
print(f"Architecture Weighted Score target (12m): {arch_target:.2f}")
print(f"Mejora relativa: {(arch_target - arch_baseline)/arch_baseline*100:.1f}%")

# ADR template migración (rellenar por estudiante)
adr_migracion = """ADR: Migración a Lakehouse Gestionado\nContexto: Necesitamos reducir time-to-data y mejorar gobernanza sin duplicar costos.\nDecisión: Migración incremental a Lakehouse (Delta/Parquet) con dual-write y validaciones.\nAlternativas: Re-host DW en cloud (menor flexibilidad), Data Mesh completo inmediato (alto riesgo).\nConsecuencias Positivas: ACID en zonas críticas, unificación batch/streaming, simplificación ETL.\nConsecuencias Negativas: Curva de aprendizaje, refactor jobs heredados.\nRiesgos: Inconsistencia temporal dual-write, subestimación costos transición.\nMitigación: Fases controladas, monitoreo KPIs, rollback plan.\n"""
print("\nADR migración:\n", adr_migracion)

# Riesgos top 5 (placeholder)
riesgos = [
    {"riesgo": "Deriva de esquemas", "prob": "Media", "impacto": "Alto", "mitigacion": "Versionado + tests schema"},
    {"riesgo": "Costos nube mayores previstos", "prob": "Alta", "impacto": "Alto", "mitigacion": "FinOps monitoreo mensual"},
    {"riesgo": "Falta adopción equipos", "prob": "Media", "impacto": "Medio", "mitigacion": "Enablement + champions"},
    {"riesgo": "Dual-write inconsistencias", "prob": "Media", "impacto": "Alto", "mitigacion": "Checks reconciliación diaria"},
    {"riesgo": "Capacidad catálogo insuficiente", "prob": "Baja", "impacto": "Medio", "mitigacion": "Escalamiento planificado"}
]
print("\nRiesgos:")
for r in riesgos:
    print(r)


<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
